# Data Prep — Smartphone (object detector)

Bangun dataset **deteksi objek smartphone** (1 kelas) dari `research/datasets/raw/phone/`
(Roboflow `gabbi/smartphone-1ldzk` v1).

Model ini BUKAN check sendiri — ia jadi salah satu komponen pipeline kompositional
`holding_phone_count`: **person+pose (`green_lane`) + smartphone (model ini) + tracker**.
Lihat `pipelines/infer.py` (Fase 2, dikerjakan di kode service).

**Output:** `research/datasets/processed/smartphone/{train,valid}/{images,labels}` + `data.yaml`.

> ⚠️ **PERINGATAN SKALA (dibuktikan di sel Profil).** Dataset raw ini mayoritas foto
> produk close-up: median bbox ≈ **17% luas frame**, dan **0%** ber-area `<0.3%` (ukuran
> HP di jarak CCTV). Model dari data ini berisiko **recall jelek di frame CCTV nyata**.
> Mitigasi di `02_train`: augmentasi scale-down kuat. Solusi sebenarnya: tambah gambar
> HP-di-genggaman dari jarak/atas. Pertimbangkan juga baseline COCO `cell phone` (id 67)
> dari weights base — sudah dilatih multi-skala, bisa jadi lebih bagus di CCTV.

## Setup

In [ ]:
%matplotlib inline
from __future__ import annotations

import random
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
import yaml
from PIL import Image

NOTEBOOK_DIR = Path.cwd()
RESEARCH_DIR = NOTEBOOK_DIR.parents[1]   # research/
AI_DIR       = NOTEBOOK_DIR.parents[2]   # ai/

RAW_DIR  = RESEARCH_DIR / 'datasets' / 'raw' / 'phone'
PROC_DIR = RESEARCH_DIR / 'datasets' / 'processed' / 'smartphone'
DATA_YAML = PROC_DIR / 'data.yaml'

CLASS_NAMES = ['smartphone']
SPLITS = ['train', 'valid']
SEED = 42

if not RAW_DIR.exists():
    raise FileNotFoundError(f'Nggak ketemu: {RAW_DIR}')
print('Raw      :', RAW_DIR)
print('Processed:', PROC_DIR)

## Konversi raw → processed

Single-class (semua → `0 = smartphone`). File di-rename PENDEK (`sp_<split><i>`) supaya
total path < 260 char (limit Windows/OneDrive + nama Roboflow panjang). `ext()` pakai
prefix `\\?\`. Set `OVERWRITE_PROCESSED=True` untuk rebuild.

In [ ]:
OVERWRITE_PROCESSED = False


def ext(p: Path) -> Path:
    return Path('\\\\?\\' + str(p.resolve()))


def poly_to_bbox(coords):
    xs, ys = coords[0::2], coords[1::2]
    xmin, xmax, ymin, ymax = min(xs), max(xs), min(ys), max(ys)
    clamp = lambda v: max(0.0, min(1.0, v))
    return clamp((xmin + xmax) / 2), clamp((ymin + ymax) / 2), clamp(xmax - xmin), clamp(ymax - ymin)


def convert_label(text):
    """Satu file label -> list 'cls cx cy w h'. polygon auto -> bbox. Semua kelas -> 0."""
    out = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        coords = [float(p) for p in parts[1:]]
        if len(coords) == 4:
            cx, cy, w, h = coords
        elif len(coords) >= 6 and len(coords) % 2 == 0:
            cx, cy, w, h = poly_to_bbox(coords)
        else:
            continue
        if w <= 0 or h <= 0:
            continue
        out.append(f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
    return out


def build():
    n = {'train': 0, 'valid': 0}
    for split in SPLITS:
        img_src = RAW_DIR / split / 'images'
        lbl_src = RAW_DIR / split / 'labels'
        if not lbl_src.exists():
            continue
        img_dst = PROC_DIR / split / 'images'
        lbl_dst = PROC_DIR / split / 'labels'
        img_dst.mkdir(parents=True, exist_ok=True)
        lbl_dst.mkdir(parents=True, exist_ok=True)
        i = 0
        for lbl in sorted(lbl_src.glob('*.txt')):
            img = None
            for suffix in ('.jpg', '.jpeg', '.png'):
                cand = img_src / (lbl.stem + suffix)
                if cand.exists():
                    img = cand
                    break
            if img is None:
                continue
            lines = convert_label(lbl.read_text(encoding='utf-8'))
            stem = f'sp_{split[0]}{i:05d}'
            ext(lbl_dst / (stem + '.txt')).write_text('\n'.join(lines) + ('\n' if lines else ''), encoding='utf-8')
            shutil.copy2(ext(img), ext(img_dst / (stem + img.suffix)))
            i += 1
            n[split] += 1
    return n


def write_data_yaml():
    cfg = {'path': str(PROC_DIR.resolve()), 'train': 'train/images', 'val': 'valid/images',
           'test': 'test/images', 'nc': len(CLASS_NAMES), 'names': CLASS_NAMES}
    PROC_DIR.mkdir(parents=True, exist_ok=True)
    DATA_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')


if DATA_YAML.exists() and not OVERWRITE_PROCESSED:
    print('processed/ sudah ada, skip build. Set OVERWRITE_PROCESSED=True untuk rebuild.')
else:
    n = build()
    write_data_yaml()
    print('Build selesai:', n)
    print('data.yaml:', DATA_YAML)

## Profil dataset — distribusi UKURAN bbox (kunci risiko CCTV)

Yang penting di sini bukan jumlah kelas (cuma 1), tapi **ukuran** bbox: kalau HP selalu
besar, model nggak akan kenal HP kecil di CCTV.

In [ ]:
rows = []
for lbl in (PROC_DIR / 'train' / 'labels').glob('*.txt'):
    for line in lbl.read_text().splitlines():
        p = line.split()
        if len(p) == 5:
            rows.append({'w': float(p[3]), 'h': float(p[4])})
bb = pd.DataFrame(rows)
bb['area'] = bb['w'] * bb['h']
n_img = {s: len(list((PROC_DIR / s / 'images').glob('*'))) for s in SPLITS}
print('Gambar per split:', n_img, '| total instance:', len(bb))
print()
print('Statistik area bbox (fraksi frame):')
print(bb[['w', 'h', 'area']].describe(percentiles=[.25, .5, .75]).round(4).to_string())
small = (bb['area'] < 0.01).mean() * 100
tiny = (bb['area'] < 0.003).mean() * 100
print(f'\nbbox <1% area (HP jauh): {small:.1f}%')
print(f'bbox <0.3% area (~jarak CCTV): {tiny:.1f}%')
print('-> kalau tiny ~0%, model HAMPIR PASTI gagal di CCTV tanpa data skala-kecil.')
print('   Mitigasi parsial: scale-down aug di 02_train. Real fix: tambah data CCTV.')

fig, ax = plt.subplots(figsize=(7, 3.5))
bb['area'].plot.hist(bins=40, ax=ax)
ax.set_title('Distribusi area bbox smartphone (train)'); ax.set_xlabel('area (fraksi frame)')
plt.tight_layout(); plt.show()

## Sample gambar + bbox

Sanity check kualitas label.

In [ ]:
N = 6
imgs = sorted((PROC_DIR / 'train' / 'images').glob('*'))
samples = random.Random(SEED).sample(imgs, min(N, len(imgs)))
cols = 3
rows_n = (len(samples) + cols - 1) // cols
fig, axes = plt.subplots(rows_n, cols, figsize=(16, 5 * rows_n))
axes = axes.flatten()
for ax, ip in zip(axes, samples):
    img = Image.open(ip); w, h = img.size; ax.imshow(img)
    lp = PROC_DIR / 'train' / 'labels' / (ip.stem + '.txt')
    if lp.exists():
        for line in lp.read_text().splitlines():
            p = line.split()
            if len(p) != 5:
                continue
            cx, cy, bw, bh = [float(v) for v in p[1:]]
            x1, y1 = (cx - bw / 2) * w, (cy - bh / 2) * h
            ax.add_patch(patches.Rectangle((x1, y1), bw * w, bh * h, lw=2,
                         edgecolor='#1f77b4', facecolor='none'))
    ax.set_title(ip.name[:40], fontsize=8); ax.axis('off')
for ax in axes[len(samples):]:
    ax.axis('off')
plt.tight_layout(); plt.show()

## Selesai

Dataset siap di `processed/smartphone/`. Lanjut **`02_train.ipynb`**.

Ingat temuan skala di atas — kalau recall di CCTV jelek, itu DATA, bukan training.